<div align="center">

<a href="https://colab.research.google.com/github/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-04/NB04_fuzzy_control.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 04: Fuzzy Logic and Fuzzy Control

**Artificial Intelligence Applications in Engineering (MUH-920 Mühendislikte Yapay Zeka Uygulamaları)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)<br>[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

<sub>Content licensed under CC BY 4.0, code under MIT. This course is updated in line with current developments in the field. Last update: October 2026.</sub>

</div>

## About this notebook

The notebook implements membership functions, fuzzy operations and Mamdani inference with NumPy, shows every inference step for one input, draws the control surface, and simulates the controller on a nonlinear tank model against an on-off controller. It then rebuilds the same controller with scikit-fuzzy [1], compares it with a zero-order Sugeno version, and closes with exercises on membership functions, centroids and rule design.

This notebook is the hands-on part of Week 4. It opens with Python step 4: NumPy arrays and first plots and continues with the sections listed in the table of contents. The concepts are explained on the [lecture page](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-04/lecture.html) and in the [PDF notes](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-04/Week04_Lecture_Notes.pdf). The [interactive lab](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-04/lab.html) holds the simulation, the self-assessment and the reflection, and the [week overview](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-04/README.md) gives the learning outcomes, the study path, the discipline challenges and the tasks.

## How to use this notebook

Run the cells in order with Shift+Enter. No earlier Python experience is assumed: Python step 4 teaches the Python of the week first, and every further piece of Python appears in a cell marked **Python move**, which explains the syntax right where it is first needed. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: Open them only after a genuine attempt. Cells that download public data need an internet connection. When a source is unreachable, the notebook falls back to a documented synthetic stand-in so that every later cell still runs.

## 0. Setup

**Python move: installing a library.** Colab has NumPy and Matplotlib but not scikit-fuzzy. A line starting with `!` runs a shell command, so `!pip install scikit-fuzzy` installs it. The `try`/`except ImportError` pattern installs it only when needed.

The last lines of the setup cell add a hint to every `NameError`. This error appears when a cell uses a name that no cell has defined in the current session, for example because the cells above have not run.

In [ ]:
try:
    import skfuzzy
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-fuzzy"], check=False)
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] {'Correct.' if ok else 'Not yet. Check the logic and run again.'}")

# Hint for a NameError: The usual cause in a notebook is a cell above that has not run in this session.
from IPython import get_ipython


def name_error_hint(shell, etype, value, tb, tb_offset=None):
    """Show a NameError as usual and add the hint."""
    shell.showtraceback((etype, value, tb), tb_offset=tb_offset)
    print("Hint: A name defined in a cell above exists only after that cell has run in this session. "
          "Runtime > Run before (Ctrl+F8) runs all cells above the current one.")


if get_ipython() is not None:
    get_ipython().set_custom_exc((NameError,), name_error_hint)

## Python step 4: NumPy arrays and first plots

> **Starting here?** Run the setup cell above first. The first code cell below checks this and stops with a message if the setup has not run in this session.

This step teaches the Python of the week with the week's own example. Read each explanation, run the cell below it and compare the output with the text; then change a value and predict the new result before running the cell again. A quick check and short exercises with immediate feedback close the step. The [Python path](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/PYTHON_PATH.md) shows how the fourteen steps of the course build on each other.

### From lists to arrays

Engineering calculations usually apply one formula to many values. NumPy provides the array, a grid of numbers of one type on which arithmetic works element by element [2]. Multiplying an array by 10 multiplies every element, whereas multiplying a list by 10 would repeat the list ten times. Arrays have a `shape`, the number of elements along each axis, and a `dtype`, the type of their elements. `np.linspace(a, b, n)` creates n evenly spaced values from a to b, which is the usual way to sample a variable such as a tank level.

In [ ]:
#@title Check that the setup cell has run
if "check" not in globals():
    raise RuntimeError("The setup cell above has not run in this session. "
                       "Run it first, then continue with this step.")

In [ ]:
import numpy as np

levels_cm = np.array([12.0, 35.5, 50.0, 71.2, 94.0])
print(levels_cm * 10)                 # millimetres, element by element
print(levels_cm.shape, levels_cm.dtype, levels_cm.mean())
x = np.linspace(0, 100, 11)           # 11 evenly spaced points from 0 to 100
print(x)

### Functions that work on arrays

A function written with array operations accepts a single number as well as a whole array. The triangular membership function of fuzzy logic rises linearly from a foot a to a peak b and falls to a second foot c. `np.minimum` and `np.maximum` compare element by element, which expresses the whole shape in one line.

In [ ]:
def tri(x, a, b, c):
    """Triangular membership function with feet a and c and peak b."""
    return np.maximum(np.minimum((x - a) / (b - a), (c - x) / (c - b)), 0.0)


print(tri(35.0, 0, 25, 50))
print(np.round(tri(x, 0, 25, 50), 2))

### Aggregation and boolean masks

Functions such as `np.sum`, `np.max` and `np.argmax` reduce an array to a single value or position. Comparisons produce boolean arrays, called masks, which count values when summed and select values when used as an index. The centroid of a fuzzy set, the defuzzification method of the Mamdani controller, is a weighted mean: The sum of x times membership is divided by the sum of membership. The example clips the output set "open" of a valve at a rule strength of 0.6.

In [ ]:
valve = np.linspace(0, 100, 201)          # valve opening in percent
open_set = tri(valve, 40, 70, 100)        # output set "open"
clipped = np.minimum(open_set, 0.6)       # a rule with strength 0.6 clips the set
centroid = np.sum(valve * clipped) / np.sum(clipped)
print(f"centroid: {centroid:.1f} % opening")
print("points above 0.5:", np.sum(clipped > 0.5))
print("openings with membership above 0.5:", valve[clipped > 0.5][[0, -1]])

The last line uses the mask as an index to keep only the matching openings, and then selects the first and the last of them.

### Plots

Matplotlib draws figures from arrays [3]. `plt.plot(x, y)` connects points with lines, the `label` argument names each curve for the legend, and axis labels with units belong on every engineering plot. The code below draws the three level sets of the tank controller.

In [ ]:
import matplotlib.pyplot as plt

level = np.linspace(0, 100, 201)
plt.figure(figsize=(6, 2.8))
plt.plot(level, tri(level, -50, 0, 50), label="low")
plt.plot(level, tri(level, 0, 50, 100), label="medium")
plt.plot(level, tri(level, 50, 100, 150), label="high")
plt.xlabel("tank level (cm)")
plt.ylabel("membership")
plt.legend()
plt.show()

Feet placed outside the range, such as -50 and 150, make the outer sets start or end at full membership at the edges of the range, which is how a controller treats a tank that is empty or full.

**Quick check.** After a = np.array([1.0, 2.0, 3.0]), what is a * 2 + 1?

A. array([3., 5., 7.])  
B. [1.0, 2.0, 3.0, 1.0, 2.0, 3.0, 1]  
C. An error  
D. 15.0

<details><summary>Show the answer</summary>

**A.** Arithmetic on arrays acts element by element. With a list, * 2 would repeat the list, and adding 1 would raise an error.

</details>

### Exercises for Python step 4

**Exercise 4.1.** Compute the membership of a level of 62 cm in the set "medium", `tri(level, 0, 50, 100)`, and store it in `py_mu_medium`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py4_mu_medium_reference():
    return min((62 - 0) / 50, (100 - 62) / 50)

In [ ]:
py_mu_medium = None
check("Exercise 4.1", py_mu_medium, _py4_mu_medium_reference())

**Exercise 4.2.** The output set `tri(valve, 40, 60, 100)` is clipped at 0.5. Compute the centroid of the clipped set and store it in `py_centroid`. Why does it lie to the right of 60?

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py4_centroid_reference():
    import numpy as np
    v = np.linspace(0, 100, 201)
    s = np.minimum(np.maximum(np.minimum((v - 40) / 20, (100 - v) / 40), 0.0), 0.5)
    return float(np.sum(v * s) / np.sum(s))

In [ ]:
py_centroid = None
check("Exercise 4.2", py_centroid, _py4_centroid_reference())

**Exercise 4.3.** Use a mask to find the share of the 201 points in `level` at which the membership of "high" exceeds 0.2, and store the share, a number between 0 and 1, in `py_share_high`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py4_share_high_reference():
    import numpy as np
    lv = np.linspace(0, 100, 201)
    hi = np.maximum(np.minimum((lv - 50) / 50, (150 - lv) / 50), 0.0)
    return float(np.mean(hi > 0.2))

In [ ]:
py_share_high = None
check("Exercise 4.3", py_share_high, _py4_share_high_reference())

## 1. Membership functions with NumPy

> **Starting here?** Run the setup cell at the top of the notebook first, or choose **Runtime > Run before** (Ctrl+F8). The first code cell below checks this and stops with a message if the setup has not run in this session.

**Python move: vectorised functions.** `trimf` works on a single number and on a whole array, because NumPy applies `np.minimum` and `np.maximum` element by element [2].

In [ ]:
#@title Check that the setup cell has run
if "check" not in globals():
    raise RuntimeError("The setup cell at the top of the notebook has not run in this session. "
                       "Run it first, or choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
def trimf(x, a, b, c):
    x = np.asarray(x, dtype=float)
    left = np.where(b > a, (x - a) / (b - a + 1e-12), 1.0)
    right = np.where(c > b, (c - x) / (c - b + 1e-12), 1.0)
    return np.clip(np.minimum(left, right), 0, 1)


def gaussmf(x, mean, sigma):
    return np.exp(-0.5 * ((np.asarray(x, dtype=float) - mean) / sigma) ** 2)


t = np.linspace(0, 40, 401)
cool, comfortable, warm = trimf(t, 5, 12, 20), trimf(t, 16, 21, 26), trimf(t, 22, 30, 38)
plt.figure(figsize=(7, 2.8))
for mu, name in [(cool, "cool"), (comfortable, "comfortable"), (warm, "warm")]:
    plt.plot(t, mu, label=name)
plt.plot(t, gaussmf(t, 21, 2.5), ls="--", label="comfortable, Gaussian")
plt.xlabel("room temperature (C)"); plt.ylabel("membership"); plt.legend(fontsize=8); plt.show()
print("23 C is comfortable to degree", float(trimf(23, 16, 21, 26)), "and warm to degree", float(trimf(23, 22, 30, 38)))

### Your turn, exercise 1

Write `trapmf(x, a, b, c, d)` for a trapezoid that rises from a to b, stays at 1 from b to c and falls from c to d. Evaluate it at x = 11 for (a, b, c, d) = (2, 5, 9, 14) and store the value in `mu_trap`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _trapmf_impl(x, a, b, c, d):
    x = np.asarray(x, dtype=float)
    return np.clip(np.minimum(np.minimum((x - a) / (b - a), 1.0), (d - x) / (d - c)), 0, 1)


def _mu_trap_reference():
    return float(_trapmf_impl(11, 2, 5, 9, 14))

In [ ]:
mu_trap = None   # your computation
check("Exercise 1", mu_trap, _mu_trap_reference())

## 2. A tank level controller, step by step

> **Starting here?** This section uses results of the cells above it, so choose **Runtime > Run before** (Ctrl+F8) first. The first code cell below checks this and stops with a message if they have not run in this session.

Inputs: level error e = set point minus level (m) and level rate r (m/s). Output: rate of change of the valve opening (1/s). Five rules follow operator logic [4, 5]:

1. IF error is low THEN open fast.
2. IF error is high THEN close fast.
3. IF error is okay AND rate is none THEN no change.
4. IF error is okay AND rate is positive THEN close slow.
5. IF error is okay AND rate is negative THEN open slow.

In [ ]:
#@title Check that the cells above have run
if not all(name in globals() for name in ["check", "trimf"]):
    raise RuntimeError("This section uses results of the cells above, which have not run in this session. "
                       "Choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
E_SETS = {"high": (-1, -1, 0), "okay": (-0.4, 0, 0.4), "low": (0, 1, 1)}
R_SETS = {"negative": (-0.1, -0.1, 0), "none": (-0.03, 0, 0.03), "positive": (0, 0.1, 0.1)}
O_SETS = {"close_fast": (-0.07, -0.05, -0.03), "close_slow": (-0.035, -0.02, -0.005), "no_change": (-0.012, 0, 0.012),
          "open_slow": (0.005, 0.02, 0.035), "open_fast": (0.03, 0.05, 0.07)}
RULES = [("low", None, "open_fast"), ("high", None, "close_fast"), ("okay", "none", "no_change"),
         ("okay", "positive", "close_slow"), ("okay", "negative", "open_slow")]
Y = np.linspace(-0.07, 0.07, 561)          # output universe


def mamdani(e, r, rules=RULES, show=False):
    e, r = float(np.clip(e, -1, 1)), float(np.clip(r, -0.1, 0.1))
    aggregated = np.zeros_like(Y)
    strengths = []
    for e_set, r_set, out in rules:
        s = float(trimf(e, *E_SETS[e_set]))
        if r_set is not None:
            s = min(s, float(trimf(r, *R_SETS[r_set])))              # AND as minimum
        strengths.append(s)
        clipped = np.minimum(s, trimf(Y, *O_SETS[out]))              # implication: clip
        aggregated = np.maximum(aggregated, clipped)                 # aggregation: max
    crisp = float(np.sum(Y * aggregated) / np.sum(aggregated)) if aggregated.sum() > 0 else 0.0
    if show:
        print("firing strengths:", [round(s, 3) for s in strengths])
        plt.figure(figsize=(6, 2.6))
        for name, p in O_SETS.items():
            plt.plot(Y, trimf(Y, *p), color="gray", lw=0.8, ls="--")
        plt.fill_between(Y, aggregated, alpha=0.5, label="aggregated")
        plt.axvline(crisp, color="C3", lw=2, label=f"centroid {crisp:.4f}")
        plt.xlabel("valve change (1/s)"); plt.legend(fontsize=8); plt.show()
    return crisp


mamdani(0.25, 0.02, show=True)

## 3. The control surface

> **Starting here?** This section uses results of the cells above it, so choose **Runtime > Run before** (Ctrl+F8) first. The first code cell below checks this and stops with a message if they have not run in this session.

**Python move: np.meshgrid.** `np.meshgrid` builds all combinations of two input grids, so the controller can be evaluated on a table of inputs and drawn as a surface.

In [ ]:
#@title Check that the cells above have run
if not all(name in globals() for name in ["check", "mamdani"]):
    raise RuntimeError("This section uses results of the cells above, which have not run in this session. "
                       "Choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
ee, rr = np.meshgrid(np.linspace(-1, 1, 41), np.linspace(-0.1, 0.1, 41))
zz = np.vectorize(mamdani)(ee, rr)
plt.figure(figsize=(5.2, 4))
plt.contourf(ee, rr, zz, levels=21, cmap="RdYlGn"); plt.colorbar(label="valve change (1/s)")
plt.xlabel("level error (m)"); plt.ylabel("level rate (m/s)"); plt.title("Mamdani control surface"); plt.show()

## 4. Closed-loop simulation

> **Starting here?** This section uses results of the cells above it, so choose **Runtime > Run before** (Ctrl+F8) first. The first code cell below checks this and stops with a message if they have not run in this session.

The tank has area A = 2 m2 and drains through an outlet with flow 0.6 sqrt(h). The valve delivers up to 1 m3/s. The set point steps from 1.0 m to 1.5 m and then to 0.8 m. The fuzzy controller changes the valve gradually; an on-off controller opens fully or closes fully.

In [ ]:
#@title Check that the cells above have run
if not all(name in globals() for name in ["check", "mamdani"]):
    raise RuntimeError("This section uses results of the cells above, which have not run in this session. "
                       "Choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
def simulate(controller, steps=1200, dt=0.5):
    h, v, prev = 1.0, 0.6, 1.0
    levels, set_points, valves = [], [], []
    for k in range(steps):
        sp = 1.0 if k < 300 else (1.5 if k < 700 else 0.8)
        e, r = sp - h, (h - prev) / dt
        prev = h
        v = controller(e, r, v, dt)
        h = max(0.0, h + (1.0 * v - 0.6 * np.sqrt(h)) * dt / 2.0)
        levels.append(h); set_points.append(sp); valves.append(v)
    return np.array(levels), np.array(set_points), np.array(valves)


fuzzy_ctrl = lambda e, r, v, dt: float(np.clip(v + mamdani(e, r) * dt, 0, 1))
on_off = lambda e, r, v, dt: 1.0 if e > 0 else 0.0
results = {"fuzzy (Mamdani)": simulate(fuzzy_ctrl), "on-off": simulate(on_off)}
time = np.arange(1200) * 0.5
fig, ax = plt.subplots(2, 1, figsize=(8, 4.6), sharex=True)
for name, (L, SP, V) in results.items():
    ax[0].plot(time, L, label=name); ax[1].plot(time, V, label=name)
    print(f"{name:16s} mean |error| {np.mean(np.abs(SP - L)):.3f} m")
ax[0].plot(time, SP, "k--", lw=0.8, label="set point"); ax[0].set_ylabel("level (m)"); ax[0].legend(fontsize=8)
ax[1].set_ylabel("valve opening"); ax[1].set_xlabel("time (s)"); plt.show()

## 5. The same controller with scikit-fuzzy

> **Starting here?** This section uses results of the cells above it, so choose **Runtime > Run before** (Ctrl+F8) first. The first code cell below checks this and stops with a message if they have not run in this session.

scikit-fuzzy describes antecedents, consequents and rules as objects [1]. Its default defuzzification is the centroid, so the result should match the from-scratch version closely.

In [ ]:
#@title Check that the cells above have run
if not all(name in globals() for name in ["check", "E_SETS", "mamdani", "O_SETS", "R_SETS", "Y"]):
    raise RuntimeError("This section uses results of the cells above, which have not run in this session. "
                       "Choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
import skfuzzy as fuzz
from skfuzzy import control as ctrl

error = ctrl.Antecedent(np.linspace(-1, 1, 401), "error")
rate = ctrl.Antecedent(np.linspace(-0.1, 0.1, 401), "rate")
valve = ctrl.Consequent(Y, "valve")
for name, p in E_SETS.items():
    error[name] = fuzz.trimf(error.universe, list(p))
for name, p in R_SETS.items():
    rate[name] = fuzz.trimf(rate.universe, list(p))
for name, p in O_SETS.items():
    valve[name] = fuzz.trimf(valve.universe, list(p))
rules = [ctrl.Rule(error["low"], valve["open_fast"]), ctrl.Rule(error["high"], valve["close_fast"]),
         ctrl.Rule(error["okay"] & rate["none"], valve["no_change"]), ctrl.Rule(error["okay"] & rate["positive"], valve["close_slow"]),
         ctrl.Rule(error["okay"] & rate["negative"], valve["open_slow"])]
sim = ctrl.ControlSystemSimulation(ctrl.ControlSystem(rules))
sim.input["error"], sim.input["rate"] = 0.25, 0.02
sim.compute()
print(f"scikit-fuzzy: {sim.output['valve']:.4f}   from scratch: {mamdani(0.25, 0.02):.4f}")

## 6. A zero-order Sugeno version

> **Starting here?** This section uses results of the cells above it, so choose **Runtime > Run before** (Ctrl+F8) first. The first code cell below checks this and stops with a message if they have not run in this session.

In a zero-order Takagi-Sugeno model, each rule outputs a constant and the result is the average of these constants weighted by the firing strengths [6]. No output shape needs to be defuzzified.

In [ ]:
#@title Check that the cells above have run
if not all(name in globals() for name in ["check", "E_SETS", "fuzzy_ctrl", "mamdani", "O_SETS", "R_SETS", "RULES", "simulate", "trimf"]):
    raise RuntimeError("This section uses results of the cells above, which have not run in this session. "
                       "Choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
CENTRES = {name: p[1] for name, p in O_SETS.items()}


def sugeno(e, r):
    e, r = float(np.clip(e, -1, 1)), float(np.clip(r, -0.1, 0.1))
    w, z = [], []
    for e_set, r_set, out in RULES:
        s = float(trimf(e, *E_SETS[e_set]))
        if r_set is not None:
            s = min(s, float(trimf(r, *R_SETS[r_set])))
        w.append(s); z.append(CENTRES[out])
    return float(np.dot(w, z) / np.sum(w)) if np.sum(w) > 0 else 0.0


L_s, SP_s, _ = simulate(lambda e, r, v, dt: float(np.clip(v + sugeno(e, r) * dt, 0, 1)))
print(f"Sugeno mean |error| {np.mean(np.abs(SP_s - L_s)):.3f} m; output at (0.25, 0.02): {sugeno(0.25, 0.02):.4f}")

### Your turn, exercise 2

Compute the Mamdani output by hand-written NumPy for e = -0.3 and r = 0.05 with the rules above, without calling `mamdani`: fuzzify, compute the five strengths, clip, aggregate and take the centroid on `Y`. Store the crisp output in `u_manual`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _u_manual_reference():
    return mamdani(-0.3, 0.05)

In [ ]:
u_manual = None   # your computation
check("Exercise 2", u_manual, _u_manual_reference(), tol=1e-4)

### Your turn, exercise 3

Make the "okay" error set narrower, (-0.2, 0, 0.2), rerun the closed-loop simulation with the Mamdani controller and store the new mean absolute error in `mae_narrow`. Does a narrower set make the controller more aggressive or more gentle near the set point?

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _mae_narrow_reference():
    global E_SETS
    saved = dict(E_SETS)
    E_SETS = dict(E_SETS, okay=(-0.2, 0, 0.2))
    L, SP, _ = simulate(fuzzy_ctrl)
    E_SETS = saved
    return float(np.mean(np.abs(SP - L)))

In [ ]:
mae_narrow = None   # your computation
check("Exercise 3", mae_narrow, _mae_narrow_reference(), tol=1e-3)

## Next steps

Take the [self-assessment](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-04/lab.html#check) in the interactive lab, then answer the [reflection prompts](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-04/lab.html#reflect) and export the learning log. The [discipline challenges](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-04/README.md#discipline-challenges), the [weekly task](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-04/README.md#weekly-task) and the [research assignment](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-04/README.md#research-and-report-assignment-optional) are listed in the week overview, and [Week 5](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-05/README.md) starts with its overview.

## References

[1] The scikit-fuzzy developers (2026). *scikit-fuzzy: Fuzzy logic toolkit for SciPy [Python package]*. <https://github.com/scikit-fuzzy/scikit-fuzzy>

[2] Harris, C. R., Millman, K. J., van der Walt, S. J., et al. (2020). Array programming with NumPy. *Nature*, *585*(7825), 357-362. <https://doi.org/10.1038/s41586-020-2649-2>

[3] Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering*, *9*(3), 90-95. <https://doi.org/10.1109/MCSE.2007.55>

[4] Mamdani, E. H., & Assilian, S. (1975). An experiment in linguistic synthesis with a fuzzy logic controller. *International Journal of Man-Machine Studies*, *7*(1), 1-13. <https://doi.org/10.1016/S0020-7373(75)80002-2>

[5] Lee, C. C. (1990). Fuzzy logic in control systems: Fuzzy logic controller, Part I. *IEEE Transactions on Systems, Man, and Cybernetics*, *20*(2), 404-418. <https://doi.org/10.1109/21.52551>

[6] Takagi, T., & Sugeno, M. (1985). Fuzzy identification of systems and its applications to modeling and control. *IEEE Transactions on Systems, Man, and Cybernetics*, *SMC-15*(1), 116-132. <https://doi.org/10.1109/TSMC.1985.6313399>